# Capítulo 3 - Exercício 3: Titanic

Neste exercício, usamos os dados do Titanic para prever a coluna `Survived`. O foco é montar um fluxo completo com pandas, pré-processamento, validação cruzada, Grid Search e avaliação final.

## Plano do exercício

1. Carregar os arquivos `train.csv`, `test.csv` e `gender_submission.csv`.
2. Separar as features (`data`) e o alvo (`label`) do conjunto de treino.
3. Definir colunas numéricas e categóricas.
4. Criar um pipeline de pré-processamento para tratar valores ausentes, escalar números e codificar categorias.
5. Treinar e comparar `RandomForestClassifier` e `KNeighborsClassifier`.
6. Usar `GridSearchCV` para ajustar hiperparâmetros.
7. Avaliar as previsões usando `gender_submission.csv` como referência didática.

In [1]:
import sys
import numpy as np
import pandas as pd

from packaging import version
import sklearn

assert sys.version_info >= (3, 7)
assert version.parse(sklearn.__version__) >= version.parse("1.0.1")

np.random.seed(42)

# Titanic - Carregando os dados

In [2]:
titanic_train = pd.read_csv("./titanic/train.csv")
titanic_test = pd.read_csv("./titanic/test.csv")
titanic_test_answer = pd.read_csv("./titanic/gender_submission.csv")

data = titanic_train.drop("Survived", axis=1)
label = titanic_train["Survived"]

## Definindo as features iniciais

In [3]:
num_features = ["Age", "SibSp", "Parch", "Fare"]
cat_features = ["Pclass", "Sex", "Embarked"]

## Pipeline de pré-processamento

In [4]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

num_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

cat_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessing = ColumnTransformer([
    ("num", num_pipeline, num_features),
    ("cat", cat_pipeline, cat_features)
])

## Modelo 1: RandomForestClassifier

In [5]:
from sklearn.ensemble import RandomForestClassifier

rnd_clf = Pipeline([
    ("preprocessing", preprocessing),
    ("classifier", RandomForestClassifier(random_state=42))
])

rnd_clf.fit(data, label);

### Validação cruzada

In [6]:
from sklearn.model_selection import cross_val_score

cross_val_score(rnd_clf, data, label, cv=3, scoring="accuracy")

array([0.77441077, 0.81818182, 0.78114478])

### Grid Search

In [7]:
from sklearn.model_selection import GridSearchCV

param_grid_rf = [
    {
        "classifier__n_estimators": [100, 200, 300],
        "classifier__max_depth": [None, 5, 10],
        "classifier__min_samples_split": [2, 5, 10],
        "classifier__min_samples_leaf": [1, 2, 4]
    }
]

grid_search_rf = GridSearchCV(rnd_clf, param_grid_rf, cv=3, scoring="accuracy")
grid_search_rf.fit(data, label);

In [8]:
print("Melhores parâmetros:", grid_search_rf.best_params_)
print("Melhor acurácia média na validação cruzada:", grid_search_rf.best_score_)

Melhores parâmetros: {'classifier__max_depth': None, 'classifier__min_samples_leaf': 4, 'classifier__min_samples_split': 2, 'classifier__n_estimators': 100}
Melhor acurácia média na validação cruzada: 0.8316498316498316


### Avaliação no arquivo de submissão

O arquivo `gender_submission.csv` não é o gabarito oficial completo do Kaggle; ele é uma referência simples baseada em gênero. Aqui ele é usado apenas para treinar o fluxo de avaliação local.

In [9]:
from sklearn.metrics import accuracy_score

best_rnd_clf = grid_search_rf.best_estimator_
test_rnd_clf = best_rnd_clf.predict(titanic_test)
accuracy_score(titanic_test_answer["Survived"], test_rnd_clf)

0.8971291866028708

## Modelo 2: KNeighborsClassifier

In [10]:
from sklearn.neighbors import KNeighborsClassifier

knn_clf = Pipeline([
    ("preprocessing", preprocessing),
    ("classifier", KNeighborsClassifier())
])

knn_clf.fit(data, label);

### Validação cruzada

In [11]:
from sklearn.model_selection import cross_val_score

cross_val_score(knn_clf, data, label, cv=5, scoring="accuracy")

array([0.78212291, 0.76966292, 0.83146067, 0.8258427 , 0.82022472])

In [12]:
from sklearn.model_selection import GridSearchCV

param_grid_knn = [{
    "classifier__n_neighbors": [3, 9, 11, 15, 20, 26, 30],
    "classifier__weights": ["uniform", "distance"],
    "classifier__p": [1, 2]
}]

grid_search_knn = GridSearchCV(
    knn_clf,
    param_grid_knn,
    cv=10,
    scoring="accuracy"
)

grid_search_knn.fit(data, label);

/home/jorge/Documentos/Projetos-Livros/Hands-on/modelos-handson/.venv/lib/python3.12/site-packages/numpy/ma/core.py:2885: RuntimeWarning: invalid value encountered in cast
  _data = np.array(data, dtype=dtype, copy=copy,


In [13]:
print("Melhores parâmetros:", grid_search_knn.best_params_)
print("Melhor acurácia média na validação cruzada:", grid_search_knn.best_score_)

Melhores parâmetros: {'classifier__n_neighbors': 20, 'classifier__p': 1, 'classifier__weights': 'uniform'}
Melhor acurácia média na validação cruzada: 0.8204744069912611


In [14]:
from sklearn.metrics import accuracy_score

best_knn_clf = grid_search_knn.best_estimator_
test_knn_clf = best_knn_clf.predict(titanic_test)
accuracy_score(titanic_test_answer["Survived"], test_knn_clf)

0.8660287081339713